# 商品匹配测试

这个 notebook 用来测试新版 RAG 商品匹配系统。

测试内容：

1. 检查配置和 Excel 文件。
2. 读取 `assets/spu.xlsx`。
3. 测试服务类型识别规则。
4. 调用 `match_product_tool` 做真实 Qwen embedding 匹配。
5. 查看本地 embedding 缓存。

注意：真实匹配会调用 Qwen embedding API。请先在 `.env` 配置：

```env
QWEN_EMBEDDING_API_KEY=你的 DashScope Key
```

如果没有配置 `QWEN_EMBEDDING_API_KEY`，代码会尝试使用 `OPENAI_API_KEY`。

In [6]:
from pathlib import Path

from config.settings import settings

print("SPU_EXCEL_PATH:", settings.spu_excel_path)
print("EMBEDDING_CACHE_DIR:", settings.embedding_cache_dir)
print("QWEN_EMBEDDING_MODEL:", settings.qwen_embedding_model)
print("QWEN_EMBEDDING_BASE_URL:", settings.qwen_embedding_base_url)
print("has_qwen_key:", bool(settings.qwen_embedding_api_key))
print("has_openai_key:", bool(settings.openai_api_key))

excel_path = Path(settings.spu_excel_path)
print("excel_exists:", excel_path.exists(), excel_path.resolve())

SPU_EXCEL_PATH: assets/spu.xlsx
EMBEDDING_CACHE_DIR: data/embedding_cache
QWEN_EMBEDDING_MODEL: text-embedding-v4
QWEN_EMBEDDING_BASE_URL: https://dashscope.aliyuncs.com/compatible-mode/v1
has_qwen_key: True
has_openai_key: True
excel_exists: True /Users/chengzongxin/project/hotel-ai-order-cursor/assets/spu.xlsx


## 1. 测试 Excel 加载

这一部分不会调用 embedding API，只检查 `assets/spu.xlsx` 是否能被正确读取，并展示前几条服务商品。

In [10]:
from rag.spu_loader import SpuExcelLoader

records = SpuExcelLoader().load()
print("record_count:", len(records))

for record in records[:5]:
    print({
        "code": record.service_product_code,
        "name": record.service_product_name,
        "raw_service_type": record.raw_service_type,
        "service_order_type": record.service_order_type,
        "category": record.category,
        "related_category": record.related_category,
        "fault": record.fault_phenomenon,
    })

record_count: 454
{'code': 'FWSP01628', 'name': '双维修', 'raw_service_type': '维修', 'service_order_type': '托管维修', 'category': '窗帘/独立窗幔', 'related_category': 'PL0126-独立窗幔', 'fault': '双故障'}
{'code': 'FWSP01627', 'name': '五金挂件（安装）', 'raw_service_type': '安装', 'service_order_type': '单次安装', 'category': '墙面地面/五金挂件', 'related_category': 'PL7304-五金挂件', 'fault': ''}
{'code': 'FWSP01623', 'name': 'LED屏', 'raw_service_type': '维修', 'service_order_type': '单次维修服务', 'category': '数码弱电/LED屏', 'related_category': 'PL7401-LED屏', 'fault': '屏幕不亮、无法上传字幕、无法更改内容或其他故障'}
{'code': 'FWSP01621', 'name': '消毒柜（大修）', 'raw_service_type': '维修', 'service_order_type': '单次维修服务', 'category': '餐饮后厨/消毒柜', 'related_category': 'PL7006-消毒柜', 'fault': '定时器不工作，臭氧发生器不工作'}
{'code': 'FWSP01620', 'name': '洗碗机（中修）', 'raw_service_type': '维修', 'service_order_type': '单次维修服务', 'category': '餐饮后厨/洗碗机', 'related_category': 'PL7007-洗碗机', 'fault': '无法进水，无法排水，漏水，水压不足，按键失灵'}


## 2. 测试服务类型识别

这一步也不会调用 embedding API。它检查用户话术能不能被规则识别成正确服务类型。

In [11]:
from rag.product_matcher import ProductMatcher

retriever = ProductMatcher()

cases = [
    "马桶堵了",
    "卫生间水龙头漏水",
    "帮我装五金挂件",
    "量一下窗帘尺寸",
    "托管维修",
]

for text in cases:
    print(text, "=>", retriever.infer_service_type_hint(text))

马桶堵了 => 单次维修服务
卫生间水龙头漏水 => 单次维修服务
帮我装五金挂件 => 单次安装
量一下窗帘尺寸 => 单次测量
托管维修 => 托管维修


## 3. 真实匹配测试

下面会调用 Qwen embedding API。

第一次运行会比较慢，因为系统会为 Excel 里的服务商品生成两组向量缓存：

- 服务商品名称
- 关联故障现象

如果结果为空，可以先把 `threshold` 从 `0.55` 降到 `0.45` 或 `0.4`。

In [12]:
from tools.product_match import match_product_tool

recall_cases = [
    {
        "query": "888房马桶堵了",
        "product": "马桶",
        "fault": "堵塞",
        "area": "卫生间",
    },
    {
        "query": "帮我装五金挂件",
        "product": "五金挂件",
        "fault": "安装",
        "area": "客房",
    },
    {
        "query": "量一下窗帘尺寸",
        "product": "窗帘",
        "fault": "测量",
        "area": "客房",
    },
]

for case in recall_cases:
    result = match_product_tool.invoke({
        **case,
        "top_k": 3,
        "threshold": 0.45,
    })
    data = result.get("data", {})
    print("\nQUERY:", case["query"])
    print("status:", result.get("status"))
    print("error_code:", result.get("error_code"))
    print("message:", result.get("message"))
    print("service_type_hint:", data.get("service_type_hint"))
    print("best_match:")
    print(data.get("best_match"))
    print("candidate_count:", data.get("count"))


QUERY: 888房马桶堵了
status: error
error_code: UPSTREAM_ERROR
message: service product recall failed: Client error '400 Bad Request' for url 'https://dashscope.aliyuncs.com/compatible-mode/v1/embeddings'
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/400
service_type_hint: None
best_match:
None
candidate_count: None

QUERY: 帮我装五金挂件
status: error
error_code: UPSTREAM_ERROR
message: service product recall failed: Client error '400 Bad Request' for url 'https://dashscope.aliyuncs.com/compatible-mode/v1/embeddings'
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/400
service_type_hint: None
best_match:
None
candidate_count: None

QUERY: 量一下窗帘尺寸
status: error
error_code: UPSTREAM_ERROR
message: service product recall failed: Client error '400 Bad Request' for url 'https://dashscope.aliyuncs.com/compatible-mode/v1/embeddings'
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/400
service_t

## 4. 查看 embedding 缓存

真实匹配运行成功后，`data/embedding_cache` 下应该会出现 `product_*.npy` 和对应的 `.json` 元数据文件。

In [5]:
cache_dir = Path(settings.embedding_cache_dir)
print("cache_dir:", cache_dir.resolve())

if cache_dir.exists():
    for path in sorted(cache_dir.glob("product_*")):
        print(path.name, path.stat().st_size, "bytes")
else:
    print("缓存目录还不存在。请先运行真实匹配测试。")

cache_dir: /Users/chengzongxin/project/hotel-ai-order-cursor/data/embedding_cache


In [13]:
from rag.spu_loader import SpuExcelLoader
from rag.product_matcher import ProductMatcher

records = SpuExcelLoader("assets/spu.xlsx").load()
print("商品数量:", len(records))
print(records[0])

retriever = ProductMatcher(excel_path="assets/spu.xlsx")

for text in ["马桶堵了", "帮我装五金挂件", "量一下窗帘尺寸", "托管维修"]:
    print(text, "=>", retriever.infer_service_type_hint(text))

商品数量: 454
ServiceProductRecord(service_product_code='FWSP01628', service_product_name='双维修', product_type='托管维修', category='窗帘/独立窗幔', raw_service_type='维修', service_order_type='托管维修', unit='个', price='0.01', price_status='正常', shelf_status='上架', repair_category='小修', related_category='PL0126-独立窗幔', related_area='客房/客房区域、客房/客房设备、客房/卫生间区域、公区/会议室、公区/设备区域、公区/操作间、公区/脏布草间、公区/楼顶、公区/电梯、公区/客房走廊、客房/维保房、公区/办公室、公区/布草间、公区/洗衣房、公区/卫生间区域、公区/健身房、公区/餐厅、公区/大堂', fault_phenomenon='双故障', display_order='0.0', remark='')
马桶堵了 => 单次维修服务
帮我装五金挂件 => 单次安装
量一下窗帘尺寸 => 单次测量
托管维修 => 托管维修


In [14]:
from tools.product_match import match_product_tool
cases = [
    {
        "query": "888房马桶堵了",
        "product": "马桶",
        "fault": "堵塞",
        "area": "卫生间",
    },
    {
        "query": "帮我装五金挂件",
        "product": "五金挂件",
        "fault": "安装",
        "area": "客房",
    },
    {
        "query": "量一下窗帘尺寸",
        "product": "窗帘",
        "fault": "测量",
        "area": "客房",
    },
]
for case in cases:
    result = match_product_tool.invoke({
        **case,
        "top_k": 3,
        "threshold": 0.45,
    })
    print("\nQUERY:", case["query"])
    print(result["data"]["service_type_hint"])
    print(result["data"]["best_match"])


QUERY: 888房马桶堵了


KeyError: 'service_type_hint'